In [ ]:
# 1) Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Set Runtime > Change runtime type > GPU')
print('GPU:', torch.cuda.get_device_name(0))
mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'VRAM: {mem_gb:.1f} GB')

## (Optional) Mount Google Drive

Recommended if you want datasets/checkpoints to persist across sessions.
If you skip this, everything is stored under `/content` and will be lost when the runtime resets.

In [ ]:
# 2) Mount Drive (optional)
from pathlib import Path
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    print('✅ Drive mounted at /content/drive')
except Exception as e:
    print('⚠️ Drive mount skipped/failed:', e)
    DRIVE_ROOT = None

## System dependencies
VLABench uses MuJoCo + EGL in headless mode.

In [ ]:
# 3) System deps
!apt-get update -qq
!apt-get install -y -qq git wget unzip ffmpeg \
  libosmesa6-dev patchelf libgl1-mesa-glx libegl1-mesa libgles2-mesa \
  build-essential
print('✅ System dependencies installed')

## Clone repositories

We clone EmbodiedLLM (this repo), then clone Evo-1 and VLABench into locations that match the wrapper script expectations.

Layout on Colab:
- `/content/EmbodiedLLM/vla-benchmark/models/Evo-1/`
- `/content/EmbodiedLLM/vla-benchmark/benchmark/VLABench/`

In [ ]:
# 4) Clone EmbodiedLLM + Evo-1 + VLABench
from pathlib import Path
import os

ROOT = Path('/content')
REPO = ROOT / 'EmbodiedLLM'

if not (REPO / '.git').exists():
    !git clone https://github.com/PrithviTM-glitch/EmbodiedLLM.git /content/EmbodiedLLM
else:
    print('✅ EmbodiedLLM already cloned')

EVO1_ROOT = REPO / 'vla-benchmark' / 'models' / 'Evo-1'
VLABENCH_ROOT = REPO / 'vla-benchmark' / 'benchmark' / 'VLABench'
EVO1_ROOT.parent.mkdir(parents=True, exist_ok=True)
VLABENCH_ROOT.parent.mkdir(parents=True, exist_ok=True)

if not (EVO1_ROOT / '.git').exists():
    !git clone https://github.com/MINT-SJTU/Evo-1.git {str(EVO1_ROOT)}
else:
    print('✅ Evo-1 already cloned')

if not (VLABENCH_ROOT / '.git').exists():
    !git clone https://github.com/OpenMOSS/VLABench.git {str(VLABENCH_ROOT)}
else:
    print('✅ VLABench already cloned')

print('EVO1_ROOT:', EVO1_ROOT)
print('VLABENCH_ROOT:', VLABENCH_ROOT)

## Python dependencies

We install:
- Evo-1 requirements
- VLABench requirements
- LeRobot pinned to v0.3.2 (matches **LeRobot v2.1 dataset format** used by Evo-1)

If you hit package conflicts, consider restarting the runtime and rerunning from the top.

In [ ]:
# 5) Install base deps + Evo-1 + VLABench + LeRobot
import os
from pathlib import Path

os.environ["MUJOCO_GL"] = "egl"

!pip install -q --upgrade pip setuptools wheel

# Colab comes with some preinstalled packages that force NumPy>=2 (jax, shap, etc).
# For this workflow we don't need them; removing avoids noisy dependency conflicts.
!pip uninstall -y -q jax jaxlib pytensor shap opencv-python opencv-python-headless opencv-contrib-python || true

# Keep NumPy in a range that also stays compatible with preinstalled TensorFlow on Colab (tf requires numpy < 2.2).
# OpenCV >=4.12 supports NumPy 2.x, so NumPy 2.1.x is a good compromise.
!pip install -q "numpy>=2.0,<2.2" pillow "opencv-python-headless>=4.12.0.88" packaging tqdm scipy colorlog colorama
!pip install -q websockets websocket-client mediapy
!pip install -q huggingface_hub

# Torch (CUDA) for Colab GPU
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121

# Evo-1 requirements
!pip install -q -r /content/EmbodiedLLM/vla-benchmark/models/Evo-1/Evo_1/requirements.txt

# FlashAttention (recommended by Evo-1; may take time)
# If it fails on your runtime, you can skip and proceed, but training quality/perf may degrade.
try:
    !MAX_JOBS=64 pip install -q -v flash-attn --no-build-isolation
    print("✅ flash-attn installed")
except Exception as e:
    print("⚠️ flash-attn install failed (continuing):", e)

# VLABench requirements
!pip install -q -r /content/EmbodiedLLM/vla-benchmark/benchmark/VLABench/requirements.txt
!pip install -q -e /content/EmbodiedLLM/vla-benchmark/benchmark/VLABench

# LeRobot (pin to v0.3.2 for v2.1 dataset compatibility)
if not Path("/content/lerobot/.git").exists():
!git clone https://github.com/huggingface/lerobot.git /content/lerobot
!cd /content/lerobot && git checkout v0.3.2
!pip install -q -e /content/lerobot

# accelerate is used by Evo-1 training
!pip install -q accelerate

print("✅ Python deps installed")

## Download VLABench assets
This is required before running trajectory generation.

In [ ]:
# 6) Download assets
from pathlib import Path
assets_dir = Path('/content/EmbodiedLLM/vla-benchmark/benchmark/VLABench/VLABench/assets')
assets_dir.mkdir(parents=True, exist_ok=True)

if (assets_dir / 'obj').exists() and (assets_dir / 'scenes').exists():
    print('✅ Assets already present; skipping download')
else:
    !cd /content/EmbodiedLLM/vla-benchmark/benchmark/VLABench && python scripts/download_assets.py
    print('✅ Assets download finished')

## Configure cache/output locations

- `HF_HOME` controls where the converted LeRobot dataset is written: `${HF_HOME}/lerobot/<dataset-name>`.
- If Drive is mounted, it’s recommended to set `HF_HOME` to Drive to keep datasets/checkpoints.

In [ ]:
# 7) Set HF_HOME (recommended: Drive)
import os
from pathlib import Path

if DRIVE_ROOT is not None:
    HF_HOME = DRIVE_ROOT / 'hf_cache'
else:
    HF_HOME = Path('/content/hf_cache')
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)
print('HF_HOME =', os.environ['HF_HOME'])

# (Optional) also store checkpoints on Drive
if DRIVE_ROOT is not None:
    CKPT_ROOT = DRIVE_ROOT / 'evo1_checkpoints'
else:
    CKPT_ROOT = Path('/content/evo1_checkpoints')
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
print('CKPT_ROOT =', CKPT_ROOT)

## Generate trajectories (HDF5)

Start small first (e.g., 1–5 samples per task) to validate your setup.

In [ ]:
# 8) Configure tasks and small-scale collection
TASKS = ['select_toy', 'select_fruit', 'select_drink']  # edit
N_SAMPLES_PER_TASK = 2  # start small; increase later
DATASET_NAME = 'vlabench_ft_demo'  # edit

print('TASKS:', TASKS)
print('N_SAMPLES_PER_TASK:', N_SAMPLES_PER_TASK)
print('DATASET_NAME:', DATASET_NAME)

In [ ]:
# 8.5) Ensure the wrapper script exists on Colab
#
# Your local workspace may have finetune_evo1_vlabench.py but the Colab clone might not.
# This cell writes a copy of the local script (embedded as base64) to the expected path before Step 9 runs it.

import base64
import subprocess
from pathlib import Path

script_path = Path('/content/EmbodiedLLM/vla-benchmark/scripts/finetune_evo1_vlabench.py')
script_path.parent.mkdir(parents=True, exist_ok=True)

# If you later push this file to GitHub, you can delete this cell and rely on `git clone` only.
SCRIPT_B64_CHUNKS = [
    '...',
]

script_bytes = base64.b64decode(''.join(SCRIPT_B64_CHUNKS))
script_path.write_bytes(script_bytes)
try:
    script_path.chmod(0o755)
except Exception:
    pass

print('✅ Wrote wrapper script:', script_path, '(bytes:', len(script_bytes), ')')

# Quick sanity check: script should parse and print help.
res = subprocess.run(['python', str(script_path), '--help'], capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError('Wrapper script sanity-check failed.\n\nstdout:\n' + res.stdout + '\n\nstderr:\n' + res.stderr)
print(res.stdout.splitlines()[0])


In [ ]:
# 9) Run the wrapper pipeline: collect + convert
# This generates HDF5 rollouts under vla-benchmark/datasets/vlabench_hdf5/
!python /content/EmbodiedLLM/vla-benchmark/scripts/finetune_evo1_vlabench.py \
  --vlabench-root /content/EmbodiedLLM/vla-benchmark/benchmark/VLABench \
  --evo1-root /content/EmbodiedLLM/vla-benchmark/models/Evo-1 \
  --tasks {' '.join(TASKS)} \
  --n-samples-per-task {N_SAMPLES_PER_TASK} \
  --dataset-name {DATASET_NAME} \
  --hf-home {str(HF_HOME)} \
  --run collect convert

## Fine-tune Evo-1

Evo-1 uses a two-stage paradigm:
- **Stage 1**: train integration module + action expert
- **Stage 2**: full-scale training (including VLM + action head)

This will take substantial time/compute. Start with smaller `max_steps` for a sanity check, then scale up.

In [ ]:
# 10) Sanity-check fine-tuning with tiny step counts (recommended first run)
STAGE1_STEPS = 50
STAGE2_STEPS = 200

!python /content/EmbodiedLLM/vla-benchmark/scripts/finetune_evo1_vlabench.py \
  --vlabench-root /content/EmbodiedLLM/vla-benchmark/benchmark/VLABench \
  --evo1-root /content/EmbodiedLLM/vla-benchmark/models/Evo-1 \
  --tasks {' '.join(TASKS)} \
  --dataset-name {DATASET_NAME} \
  --hf-home {str(HF_HOME)} \
  --run stage1 stage2 \
  --use-deepspeed \
  --save-dir-stage1 {str(CKPT_ROOT / 'stage1')} \
  --save-dir-stage2 {str(CKPT_ROOT / 'stage2')} \
  --stage1-max-steps {STAGE1_STEPS} \
  --stage2-max-steps {STAGE2_STEPS}

## Full training (optional)

Once the sanity check works, you can run the full default steps (Stage1=5000, Stage2=80000).
You may also want to increase dataset size and/or number of tasks.

In [ ]:
# 11) Full training (long run) - uncomment when ready
# !python /content/EmbodiedLLM/vla-benchmark/scripts/finetune_evo1_vlabench.py \
#   --vlabench-root /content/EmbodiedLLM/vla-benchmark/benchmark/VLABench \
#   --evo1-root /content/EmbodiedLLM/vla-benchmark/models/Evo-1 \
#   --tasks {' '.join(TASKS)} \
#   --n-samples-per-task 50 \
#   --dataset-name {DATASET_NAME} \
#   --hf-home {str(HF_HOME)} \
#   --run collect convert stage1 stage2 \
#   --use-deepspeed \
#   --save-dir-stage1 {str(CKPT_ROOT / 'stage1_full')} \
#   --save-dir-stage2 {str(CKPT_ROOT / 'stage2_full')}